In [1]:
import sys
sys.path.append("..")

In [2]:
from openai import OpenAI

# Basic

In [3]:
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # required by SDK, ignored by Ollama
)

In [4]:
messages = [
    {
        "role": "system",
        "content": "Kamu adalah chatbot yang ramah"
    },
    {
        "role": "user",
        "content": "Halo"
    }
]

In [5]:
response = client.chat.completions.create(
    model="qwen3.5:9b",
    max_tokens=4096,
    messages=messages,
)

In [6]:
response.choices[0].message

ChatCompletionMessage(content='Halo! 👋 Senang sekali bertemu Anda di sini. Ada yang bisa saya bantu hari ini? Semoga harimu menyenangkan! 😊', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='Thinking Process:\n\n1.  **Analyze the Request:**\n    *   Input: "Halo" (Indonesian for "Hello").\n    *   Context: I am a friendly chatbot.\n    *   Goal: Respond in Indonesian, maintaining a friendly and helpful tone.\n\n2.  **Determine the Appropriate Response:**\n    *   Greeting: "Halo!" or "Hai!"\n    *   Tone: Friendly, warm, welcoming.\n    *   Content: Acknowledge the greeting, offer assistance.\n    *   Language: Indonesian (matching the user\'s input).\n\n3.  **Drafting the Response:**\n    *   Option 1 (Simple): Halo! Apa kabar? Ada yang bisa saya bantu hari ini?\n    *   Option 2 (Warm & Engaging): Halo! Senang bertemu Anda. Bagaimana hari ini? Ada yang ingin ditanya atau butuhkan bantuan?\n    *   Option 3 (Friendly Chatbot

# tool calling

## Prepare Database & Schemas

In [1]:
import sqlite3
import pandas as pd

# Buat koneksi
conn = sqlite3.connect("products.db")
cursor = conn.cursor()

# Buat table products
cursor.execute("""
CREATE TABLE IF NOT EXISTS products (
    product_id    TEXT PRIMARY KEY,
    product_name  TEXT NOT NULL,
    product_type  TEXT,
    description   TEXT,
    price         REAL NOT NULL,
    stock         INTEGER DEFAULT 0
)
""")
conn.commit()
print("✅ Table 'products' created")

✅ Table 'products' created


## Insert Sample Data

In [2]:
# Simulasi data
sample_data = pd.DataFrame([
    {"product_id": "SKU001", "product_name": "Nike Air Max 270",     "product_type": "Sepatu",      "description": "Sepatu lari ringan dengan bantalan Air Max untuk kenyamanan maksimal", "price": 2199000, "stock": 45},
    {"product_id": "SKU002", "product_name": "Adidas Ultraboost 23", "product_type": "Sepatu",      "description": "Sepatu lari premium dengan teknologi Boost untuk pengembalian energi",  "price": 2899000, "stock": 30},
    {"product_id": "SKU003", "product_name": "Uniqlo Airism T-Shirt","product_type": "Pakaian",     "description": "Kaos dengan teknologi pendingin, cepat kering, anti bau",               "price": 199000,  "stock": 200},
    {"product_id": "SKU004", "product_name": "Samsung Galaxy S24",   "product_type": "Elektronik",  "description": "Smartphone flagship dengan AI Galaxy, kamera 200MP, layar AMOLED 6.2 inch", "price": 13999000, "stock": 15},
    {"product_id": "SKU005", "product_name": "Sony WH-1000XM5",      "product_type": "Elektronik",  "description": "Headphone wireless noise cancelling terbaik dengan 30 jam battery life",  "price": 4999000, "stock": 25},
    {"product_id": "SKU006", "product_name": "Erigo Flannel Shirt",  "product_type": "Pakaian",     "description": "Kemeja flannel kotak-kotak casual, bahan katun premium",                 "price": 259000,  "stock": 80},
])

# insert to database
sample_data.to_sql("products", conn, if_exists="replace", index=False)
print(f"✅ {len(sample_data)} products inserted")

# verifikasi
pd.read_sql("SELECT product_id, product_name, price FROM products", conn)

✅ 6 products inserted


,product_id,product_name,price
0,SKU001,Nike Air Max 270,2199000
1,SKU002,Adidas Ultraboost 23,2899000
2,SKU003,Uniqlo Airism T-Shirt,199000
3,SKU004,Samsung Galaxy S24,13999000
4,SKU005,Sony WH-1000XM5,4999000
5,SKU006,Erigo Flannel Shirt,259000


## function tools

In [3]:
import json

def search_product(conn, query: str) -> str:
    """
    Cari produk berdasarkan nama (fuzzy/partial match).
    Mengembalikan JSON string agar mudah dikonsumsi LLM.
    """
    sql = """
        SELECT product_id, product_name, product_type, description, price, stock
        FROM products
        WHERE LOWER(product_name) LIKE LOWER(?)
           OR LOWER(description) LIKE LOWER(?)
           OR LOWER(product_type) LIKE LOWER(?)
    """
    wildcard = f"%{query}%"
    rows = conn.execute(sql, (wildcard, wildcard, wildcard)).fetchall()
    
    if not rows:
        return json.dumps({"found": False, "message": f"Tidak ada produk yang cocok dengan '{query}'"})
    
    columns = ["product_id", "product_name", "product_type", "description", "price", "stock"]
    results = [dict(zip(columns, row)) for row in rows]
    
    # Format harga agar jelas
    for r in results:
        r["price_formatted"] = f"Rp {r['price']:,.0f}"
    
    return json.dumps({"found": True, "count": len(results), "products": results}, ensure_ascii=False)


def get_product_by_id(conn, product_id: str) -> str:
    """Ambil detail produk berdasarkan ID yang tepat."""
    row = conn.execute(
        "SELECT product_id, product_name, product_type, description, price, stock FROM products WHERE product_id = ?",
        (product_id,)
    ).fetchone()
    
    if not row:
        return json.dumps({"found": False, "message": f"Product ID '{product_id}' tidak ditemukan"})
    
    columns = ["product_id", "product_name", "product_type", "description", "price", "stock"]
    result = dict(zip(columns, row))
    result["price_formatted"] = f"Rp {result['price']:,.0f}"
    return json.dumps({"found": True, "product": result}, ensure_ascii=False)


# ── Test ──
print(search_product(conn, "nike"))

{"found": true, "count": 1, "products": [{"product_id": "SKU001", "product_name": "Nike Air Max 270", "product_type": "Sepatu", "description": "Sepatu lari ringan dengan bantalan Air Max untuk kenyamanan maksimal", "price": 2199000, "stock": 45, "price_formatted": "Rp 2,199,000"}]}


## Define Tool Schema

In [4]:
# ── Tool definitions (OpenAI function calling format) ──
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_product",
            "description": (
                "Cari produk berdasarkan nama, tipe, atau kata kunci dari deskripsi. "
                "Gunakan tool ini ketika customer menanyakan tentang produk tertentu, "
                "mencari rekomendasi, atau ingin tahu produk apa saja yang tersedia."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Kata kunci pencarian: nama produk, tipe (sepatu/elektronik/pakaian), atau kata dari deskripsi"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_product_by_id",
            "description": (
                "Ambil detail lengkap satu produk berdasarkan product_id (contoh: SKU001). "
                "Gunakan ini ketika sudah tahu product_id yang tepat."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "product_id": {
                        "type": "string",
                        "description": "ID produk, format SKUxxx"
                    }
                },
                "required": ["product_id"]
            }
        }
    }
]

print("✅ Tools defined:", [t["function"]["name"] for t in tools])

✅ Tools defined: ['search_product', 'get_product_by_id']


## System Prompt

In [5]:
SYSTEM_PROMPT = """Kamu adalah customer service AI untuk toko online kami.

ATURAN PENTING:
1. SELALU gunakan tool search_product atau get_product_by_id untuk menjawab pertanyaan tentang produk.
2. JANGAN PERNAH mengarang harga, stok, atau detail produk dari pengetahuanmu sendiri.
3. Jika tool mengembalikan hasil, sampaikan informasi PERSIS seperti data yang dikembalikan — terutama HARGA.
4. Jika produk tidak ditemukan, katakan dengan sopan bahwa produk tidak tersedia dalam katalog.
5. Untuk pertanyaan umum (salam, cara order, dll), jawab secara natural tanpa tool.
6. Selalu gunakan format harga Rupiah yang sudah disediakan (price_formatted).
7. Jawab dalam Bahasa Indonesia yang ramah dan profesional.
"""

## Agent Loop

In [12]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # required by SDK, ignored by Ollama
)

# ── Mapping nama tool ke fungsi Python ──
TOOL_FUNCTIONS = {
    "search_product": lambda args: search_product(conn, args["query"]),
    "get_product_by_id": lambda args: get_product_by_id(conn, args["product_id"]),
}


def chat_with_agent(user_message: str, conversation_history: list) -> str:
    """
    Satu turn percakapan dengan agent.
    Handle: user message → LLM → (optional tool call → tool result → LLM) → final response
    """
    # Tambahkan pesan user ke history
    conversation_history.append({"role": "user", "content": user_message})
    
    # ── Panggil OpenAI ──
    response = client.chat.completions.create(
        model="qwen3.5:9b",  # Atau "gpt-4o" untuk akurasi lebih tinggi
        messages=[{"role": "system", "content": SYSTEM_PROMPT}] + conversation_history,
        tools=tools,
        tool_choice="auto",  # Biarkan LLM memutuskan kapan pakai tool
    )
    
    assistant_message = response.choices[0].message
    
    # ── Cek apakah LLM mau memanggil tool ──
    if assistant_message.tool_calls:
        # Simpan response LLM (berisi tool_calls) ke history
        conversation_history.append(assistant_message)
        
        # Eksekusi SETIAP tool call
        for tool_call in assistant_message.tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)
            
            print(f"   🔧 Tool called: {fn_name}({fn_args})")
            
            # Jalankan fungsi yang sesuai
            result = TOOL_FUNCTIONS[fn_name](fn_args)
            
            print(f"   📦 Result: {result[:100]}...")
            
            # Kirim hasil tool kembali ke LLM
            conversation_history.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            })
        
        # ── Panggil LLM lagi dengan hasil tool ──
        response = client.chat.completions.create(
            model="qwen3.5:9b",
            messages=[{"role": "system", "content": SYSTEM_PROMPT}] + conversation_history,
            tools=tools,
        )
        assistant_message = response.choices[0].message
    
    # Simpan jawaban final ke history
    conversation_history.append({"role": "assistant", "content": assistant_message.content})
    
    return assistant_message.content

In [13]:
history = []

# ── Test 1: Pertanyaan harga spesifik ──
print("=" * 60)
print("USER: Berapa harga Nike Air Max?")
print("=" * 60)
response = chat_with_agent("Berapa harga Nike Air Max?", history)
print(f"\n🤖 AGENT:\n{response}")

USER: Berapa harga Nike Air Max?
   🔧 Tool called: search_product({'query': 'Nike Air Max'})
   📦 Result: {"found": true, "count": 1, "products": [{"product_id": "SKU001", "product_name": "Nike Air Max 270"...

🤖 AGENT:
Halo! Berdasarkan katalog kami saat ini, **Nike Air Max 270** tersedia dengan harga **Rp 2,199,000**. 

Detail produk:
- **Nama:** Nike Air Max 270
- **Tipe:** Sepatu
- **Deskripsi:** Sepatu lari ringan dengan bantalan Air Max untuk kenyamanan maksimal
- **Stok:** 45 unit

Apakah Anda ingin memesan atau ada pertanyaan lain tentang produk ini? 😊


In [14]:
# ── Test 2: Pencarian berdasarkan kategori ──
print("\n" + "=" * 60)
print("USER: Ada sepatu apa aja?")
print("=" * 60)
response = chat_with_agent("Ada sepatu apa aja?", history)
print(f"\n🤖 AGENT:\n{response}")


USER: Ada sepatu apa aja?
   🔧 Tool called: search_product({'query': 'sepatu'})
   📦 Result: {"found": true, "count": 2, "products": [{"product_id": "SKU001", "product_name": "Nike Air Max 270"...

🤖 AGENT:
Halo! 🛍️ Kami memiliki **sepatu** berikut yang tersedia di katalog kami saat ini:

---

### **1. Nike Air Max 270** - `SKU001`
- **Harga:** Rp 2,199,000
- **Stok:** 45 unit
- **Deskripsi:** Sepatu lari ringan dengan bantalan Air Max untuk kenyamanan maksimal

---

### **2. Adidas Ultraboost 23** - `SKU002`
- **Harga:** Rp 2,899,000
- **Stok:** 30 unit
- **Deskripsi:** Sepatu lari premium dengan teknologi Boost untuk pengembalian energi

---

Selain yang di atas, apakah Anda ingin melihat produk lain seperti pakaian atau elektronik? Atau ada sepatu tertentu yang sedang Anda cari? 😊


In [15]:
# ── Test 4: Pertanyaan konseptual ──
print("\n" + "=" * 60)
print("USER: Ada headphone yang battery-nya tahan lama?")
print("=" * 60)
response = chat_with_agent("Ada headphone yang battery-nya tahan lama?", history)
print(f"\n🤖 AGENT:\n{response}")


USER: Ada headphone yang battery-nya tahan lama?
   🔧 Tool called: search_product({'query': 'headphone'})
   📦 Result: {"found": true, "count": 1, "products": [{"product_id": "SKU005", "product_name": "Sony WH-1000XM5",...

🤖 AGENT:
Halo! ☎️ Berikut **headphone dengan baterai tahan lama** yang kami tawarkan:

---

### **Sony WH-1000XM5** — `SKU005`
- **Harga:** Rp 4,999,000
- **Stok:** 25 unit
- **Tipe:** Elektronik (Headphone)
- **Deskripsi:** Headphone wireless noise cancelling terbaik dengan **30 jam battery life**

---

Ini adalah headphone premium dengan fitur *noise cancelling* dan baterai yang bisa bertahan hingga **30 jam penggunaan**! 

Apakah Anda tertarik untuk memesan atau ada pertanyaan lain tentang spesifikasi teknisnya? 😊


In [16]:
# ── Test 4: Pertanyaan konseptual ──
print("\n" + "=" * 60)
print("USER: Apakah ada baju anak umur 5 tahun?")
print("=" * 60)
response = chat_with_agent("Apakah ada baju anak umur 5 tahun?", history)
print(f"\n🤖 AGENT:\n{response}")


USER: Apakah ada baju anak umur 5 tahun?
   🔧 Tool called: search_product({'query': 'baju anak'})
   📦 Result: {"found": false, "message": "Tidak ada produk yang cocok dengan 'baju anak'"}...

🤖 AGENT:
Maaf, saat ini **kami belum memiliki produk baju anak usia 5 tahun** di katalog kami. 📦

Katalog kami fokus pada:
- 🏃‍♂️ Sepatu (Nike & Adidas)
- 🔊 Elektronik (Headphone)
- 👔 Pakaian dewasa lainnya

Jika Anda tertarik dengan pakaian untuk kebutuhan lain, silakan tanyakan lebih detail mengenai tipe atau bahan yang diinginkan! 😊

Atau mungkin Anda ingin melihat produk **pakaian dewasa** terlebih dahulu?
